# 00 - Setup del entorno y descarga de datos

Deja `data/raw/` con la misma muestra de datos en cualquier maquina, para que los notebooks 01-05 sean reproducibles.

## Por que una muestra y no el dataset completo

El dataset completo son 158 GB (68 archivos de `train_landmarks/*.parquet`, ~1.5 GB c/u). `train.csv` ya trae la metadata de las 67,208 secuencias; con 2 archivos de landmarks alcanza para el EDA.

Se descarga con la CLI de `kaggle` y no `kagglehub`: para archivos individuales, `kagglehub` devuelve el contenido comprimido en zip pero con el nombre sin `.zip`, lo que rompe `pd.read_csv`/`read_parquet` en silencio.
- `train.csv`, `supplemental_metadata.csv`, `character_to_prediction_index.json` (pocos MB)
- 2 archivos fijos de `train_landmarks/` (`config.SAMPLE_LANDMARK_PATHS`)

In [1]:
import sys
sys.path.append("..")

from pathlib import Path
from src import config


## 1. Credenciales de Kaggle

Requiere `~/.kaggle/kaggle.json` (API token de https://www.kaggle.com/settings) y haber aceptado las reglas de la competencia: https://www.kaggle.com/competitions/asl-fingerspelling/rules

In [2]:
!pip install -q kaggle


## 2. Descargar metadata completa (train.csv y compania)

Estos archivos vienen comprimidos como `<nombre>.zip` aunque se pida solo uno; `unzip -o` los deja listos y `rm` limpia el zip.

In [ ]:
import os
import getpass

# Por si tenemos problemas al ingresar con el token de forma segura. 
# No quedará guardado en el texto del notebook.
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass(prompt='Pega tu token de Kaggle aquí y presiona Enter: ')

In [9]:
config.DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

!kaggle competitions download -c asl-fingerspelling -f train.csv -p "{config.DATA_RAW_DIR}"
!kaggle competitions download -c asl-fingerspelling -f supplemental_metadata.csv -p "{config.DATA_RAW_DIR}"
!kaggle competitions download -c asl-fingerspelling -f character_to_prediction_index.json -p "{config.DATA_RAW_DIR}"

train.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
supplemental_metadata.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
character_to_prediction_index.json: Skipping, found more recently modified local copy (use --force to force download)


In [11]:
import zipfile

# 1. Descomprimir y borrar train.csv.zip
train_zip = config.DATA_RAW_DIR / "train.csv.zip"
if train_zip.exists():
    with zipfile.ZipFile(train_zip, 'r') as zip_ref:
        zip_ref.extractall(config.DATA_RAW_DIR)
    train_zip.unlink() # Esto reemplaza al comando 'rm'

# 2. Descomprimir y borrar supplemental_metadata.csv.zip
supp_zip = config.DATA_RAW_DIR / "supplemental_metadata.csv.zip"
if supp_zip.exists():
    with zipfile.ZipFile(supp_zip, 'r') as zip_ref:
        zip_ref.extractall(config.DATA_RAW_DIR)
    supp_zip.unlink()

## 3. Descargar la muestra fija de landmarks

`config.SAMPLE_LANDMARK_PATHS` es la lista fija de archivos de muestra.

In [12]:
import zipfile
from pathlib import Path

landmarks_dir = config.train_landmarks_dir()
landmarks_dir.mkdir(parents=True, exist_ok=True)

for rel_path in config.SAMPLE_LANDMARK_PATHS:
    fname = Path(rel_path).name
    if (landmarks_dir / fname).exists():
        print("ya existe:", fname)
        continue
    
    # Descarga con comillas en la ruta para evitar el error de los espacios
    !kaggle competitions download -c asl-fingerspelling -f {rel_path} -p "{landmarks_dir}"
    
    # Descomprimir y borrar usando Python en lugar de unzip y rm
    zip_path = landmarks_dir / f"{fname}.zip"
    if zip_path.exists():
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(landmarks_dir)
        zip_path.unlink()


  0%|          | 0.00/1.27G [00:00<?, ?B/s]
  0%|          | 1.00M/1.27G [00:00<12:42, 1.79MB/s]
  0%|          | 2.00M/1.27G [00:00<06:32, 3.47MB/s]
  0%|          | 5.00M/1.27G [00:00<02:44, 8.27MB/s]
  1%|          | 10.0M/1.27G [00:00<01:18, 17.2MB/s]
  1%|          | 13.0M/1.27G [00:01<01:48, 12.5MB/s]
  1%|          | 15.0M/1.27G [00:01<01:38, 13.8MB/s]
  1%|▏         | 17.0M/1.27G [00:03<05:54, 3.81MB/s]
  2%|▏         | 26.0M/1.27G [00:03<02:20, 9.54MB/s]
  2%|▏         | 32.0M/1.27G [00:03<01:35, 13.9MB/s]
  3%|▎         | 37.0M/1.27G [00:03<01:50, 12.0MB/s]
  3%|▎         | 41.0M/1.27G [00:04<01:32, 14.3MB/s]
  4%|▍         | 49.0M/1.27G [00:04<01:00, 21.7MB/s]
  4%|▍         | 54.0M/1.27G [00:04<01:09, 18.8MB/s]
  5%|▍         | 59.0M/1.27G [00:04<00:59, 22.0MB/s]
  5%|▍         | 63.0M/1.27G [00:04<00:54, 23.7MB/s]
  5%|▌         | 71.0M/1.27G [00:04<00:39, 32.8MB/s]
  6%|▌         | 76.0M/1.27G [00:05<00:40, 31.8MB/s]
  6%|▌         | 80.0M/1.27G [00:05<00:40, 31.7MB/s]
 

## 4. Verificar el contenido descargado

In [13]:
for p in sorted(config.DATA_RAW_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(config.DATA_RAW_DIR), f"({p.stat().st_size / 1e6:.1f} MB)")


.gitkeep (0.0 MB)
character_to_prediction_index.json (0.0 MB)
supplemental_metadata.csv (5.1 MB)
train.csv (5.2 MB)
train_landmarks\1019715464.parquet (1536.0 MB)
train_landmarks\1021040628.parquet (1518.8 MB)


## 5. Probar la carga con `src/data_loading.py`

Si esto corre sin errores, el resto de notebooks (01-05) ya pueden usar `dl.load_train_index()` / `dl.load_landmarks(dl.landmark_path(...))` sin configuracion adicional.

In [14]:
from src import data_loading as dl

train_df = dl.load_train_index()
print(train_df.shape)

sample_landmarks = dl.load_landmarks(dl.landmark_path(config.SAMPLE_LANDMARK_PATHS[0]))
sample_landmarks.shape


(67208, 5)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.